In [ ]:
# Complete audio analysis function using NumPy

import numpy as np



def analyze_audio(audio_data, sample_rate):

"""

Analyze audio and return key properties.



Parameters:

- audio_data: Raw bytes from recording

- sample_rate: Samples per second (e.g., 16000)



Returns:

- Dictionary containing duration, avg_volume, max_volume, samples

"""



# Convert raw bytes to NumPy array of 16-bit integers

# dtype=np.int16 means each number is a 16-bit signed integer

samples = np.frombuffer(audio_data, dtype=np.int16)



# Calculate duration: total samples divided by samples per second

duration = len(samples) / sample_rate



# Calculate average amplitude using absolute values

avg_volume = np.mean(np.abs(samples))



# Calculate maximum amplitude

max_volume = np.max(np.abs(samples))



# Return all measurements in a dictionary

return {

     'duration': duration,

     'avg_volume': avg_volume,

     'max_volume': max_volume,

     'samples': samples  # Keep for visualization

}

In [ ]:
"""

L2: Voice Analysis Lab

Record → Analyze → Compare Two Recordings



============== DEPENDENCY SETUP ==============

CHECK IF INSTALLED: pip show SpeechRecognition pyaudio numpy matplotlib



INSTALL - WINDOWS:

pip install SpeechRecognition pyaudio numpy matplotlib



INSTALL - macOS:

brew install portaudio

pip install SpeechRecognition pyaudio numpy matplotlib

==============================================

"""



import threading

import sys



# Dependency check

try:

import pyaudio

import numpy as np

import matplotlib.pyplot as plt

import speech_recognition as sr

from speech_recognition import AudioData

except ImportError as e:

print(f"❌ Missing library: {e.name}")

print("\n???? Install commands:")

print("   Windows: pip install SpeechRecognition pyaudio numpy matplotlib")

print("   macOS:   brew install portaudio && pip install SpeechRecognition pyaudio numpy matplotlib")

sys.exit(1)



stop_event = threading.Event()



def wait_for_enter():

input()

stop_event.set()



def record_audio(label):

stop_event.clear()  # Reset for new recording

p = pyaudio.PyAudio()

stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,

                 input=True, frames_per_buffer=1024)

frames = []



print(f"\n???? {label}")

print("   Press Enter to stop...")

threading.Thread(target=wait_for_enter, daemon=True).start()



print("???? Recording", end="", flush=True)

while not stop_event.is_set():

     frames.append(stream.read(1024, exception_on_overflow=False))

     print(".", end="", flush=True)

print(" ✅")



stream.stop_stream()

stream.close()

width = p.get_sample_size(pyaudio.paInt16)

p.terminate()

return b''.join(frames), 16000, width



def analyze_audio(data, rate):

samples = np.frombuffer(data, dtype=np.int16)

return {

     'duration': len(samples) / rate,

     'avg_volume': np.mean(np.abs(samples)),

     'max_volume': np.max(np.abs(samples)),

     'samples': samples

}



def transcribe(data, rate, width):

recognizer = sr.Recognizer()

try:

     return recognizer.recognize_google(AudioData(data, rate, width))

except:

     return "[Could not transcribe]"



def display_stats(stats, text, label):

print(f"\n{'─' * 40}")

print(f"???? {label}")

print(f"{'─' * 40}")

print(f"⏱️  Duration:  {stats['duration']:.2f} seconds")

print(f"???? Avg Amplitude: {stats['avg_volume']:.0f}")

print(f"???? Max Amplitude: {stats['max_volume']:.0f}")

print(f"???? Transcription: {text}")



def compare(stats1, stats2):

print("\n" + "=" * 40)

print("???? COMPARISON RESULTS")

print("=" * 40)



# Duration comparison

if stats1['duration'] > stats2['duration']:

     longer = "Recording 1"

     diff = ((stats1['duration'] - stats2['duration']) / stats2['duration']) * 100

else:

     longer = "Recording 2"

     diff = ((stats2['duration'] - stats1['duration']) / stats1['duration']) * 100

print(f"⏱️  {longer} is longer by {diff:.1f}%")



# Volume comparison

if stats1['avg_volume'] > stats2['avg_volume']:

     louder = "Recording 1"

     diff = ((stats1['avg_volume'] - stats2['avg_volume']) / stats2['avg_volume']) * 100

else:

     louder = "Recording 2"

     diff = ((stats2['avg_volume'] - stats1['avg_volume']) / stats1['avg_volume']) * 100

print(f"???? {louder} is louder by {diff:.1f}%")



def plot_both(stats1, stats2, rate):

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))



# Plot Recording 1

t1 = np.linspace(0, len(stats1['samples']) / rate, len(stats1['samples']))

ax1.plot(t1, stats1['samples'], color='blue', linewidth=0.5)

ax1.set_title(f"Recording 1 (Normal) - Duration: {stats1['duration']:.2f}s, Avg: {stats1['avg_volume']:.0f}")

ax1.set_ylabel("Amplitude")

ax1.grid(True, alpha=0.3)

ax1.set_ylim(-35000, 35000)



# Plot Recording 2

t2 = np.linspace(0, len(stats2['samples']) / rate, len(stats2['samples']))

ax2.plot(t2, stats2['samples'], color='red', linewidth=0.5)

ax2.set_title(f"Recording 2 (Modified) - Duration: {stats2['duration']:.2f}s, Avg: {stats2['avg_volume']:.0f}")

ax2.set_xlabel("Time (seconds)")

ax2.set_ylabel("Amplitude")

ax2.grid(True, alpha=0.3)

ax2.set_ylim(-35000, 35000)



plt.tight_layout()

plt.show()



def main():

print("=" * 40)

print("???? VOICE ANALYSIS LAB")

print("=" * 40)

print("Record twice and compare your voice!")



# Recording 1: Control

audio1, rate, width = record_audio("Recording 1: Speak NORMALLY")

stats1 = analyze_audio(audio1, rate)

text1 = transcribe(audio1, rate, width)

display_stats(stats1, text1, "Recording 1 Results")



# Prompt for Recording 2

input("\n???? Press Enter, then speak LOUDER or FASTER...")



# Recording 2: Variable

audio2, rate, width = record_audio("Recording 2: CHANGE your voice!")

stats2 = analyze_audio(audio2, rate)

text2 = transcribe(audio2, rate, width)

display_stats(stats2, text2, "Recording 2 Results")



# Compare and visualize

compare(stats1, stats2)

plot_both(stats1, stats2, rate)



if __name__ == "__main__":

main()

In [ ]:
!pip install SpeechRecognition
import numpy as np
import matplotlib.pyplot as plt
import speech_recognition as sr
from speech_recognition import AudioData
from IPython.display import Javascript, display, Audio
from google.colab import output
from base64 import b64decode
import wave

# ---------------- RECORD FUNCTION ----------------
def record_audio_colab(label, seconds=5):
    print(f"\n🎤 {label}")
    print(f"Recording for {seconds} seconds...")

    js = f"""
    async function recordAudio() {{
      const stream = await navigator.mediaDevices.getUserMedia({{audio:true}});
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, {seconds * 1000}));

      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);

      const blob = new Blob(chunks);
      const arrayBuffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(arrayBuffer);

      let binary = '';
      bytes.forEach(b => binary += String.fromCharCode(b));
      return btoa(binary);
    }}
    recordAudio();
    """

    audio_data = output.eval_js(js)
    audio_bytes = b64decode(audio_data)

    with open("temp.webm", "wb") as f:
        f.write(audio_bytes)

    # Convert to WAV
    !ffmpeg -i temp.webm -ac 1 -ar 16000 -y output.wav > /dev/null 2>&1

    return "output.wav"


# ---------------- LOAD AUDIO ----------------
def read_wav(filename):
    with wave.open(filename, 'rb') as wf:
        rate = wf.getframerate()
        width = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    return frames, rate, width


# ---------------- ANALYSIS ----------------
def analyze_audio(data, rate):
    samples = np.frombuffer(data, dtype=np.int16)
    return {
        'duration': len(samples) / rate,
        'avg_volume': np.mean(np.abs(samples)),
        'max_volume': np.max(np.abs(samples)),
        'samples': samples
    }


# ---------------- TRANSCRIBE ----------------
def transcribe(data, rate, width):
    recognizer = sr.Recognizer()
    try:
        return recognizer.recognize_google(AudioData(data, rate, width))
    except:
        return "[Could not transcribe]"


# ---------------- DISPLAY ----------------
def display_stats(stats, text, label):
    print("\n" + "─" * 40)
    print(f"📊 {label}")
    print("─" * 40)
    print(f"⏱️ Duration: {stats['duration']:.2f} sec")
    print(f"🔊 Avg Volume: {stats['avg_volume']:.0f}")
    print(f"📈 Max Volume: {stats['max_volume']:.0f}")
    print(f"📝 Text: {text}")


# ---------------- COMPARE ----------------
def compare(stats1, stats2):
    print("\n" + "=" * 40)
    print("📊 COMPARISON")
    print("=" * 40)

    if stats1['duration'] > stats2['duration']:
        print("Recording 1 is longer")
    else:
        print("Recording 2 is longer")

    if stats1['avg_volume'] > stats2['avg_volume']:
        print("Recording 1 is louder")
    else:
        print("Recording 2 is louder")


# ---------------- PLOT ----------------
def plot_both(stats1, stats2, rate):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

    t1 = np.linspace(0, len(stats1['samples']) / rate, len(stats1['samples']))
    ax1.plot(t1, stats1['samples'])
    ax1.set_title("Recording 1")

    t2 = np.linspace(0, len(stats2['samples']) / rate, len(stats2['samples']))
    ax2.plot(t2, stats2['samples'])
    ax2.set_title("Recording 2")

    plt.tight_layout()
    plt.show()


# ---------------- MAIN ----------------
def main():
    print("=" * 40)
    print("🎙️ VOICE ANALYSIS LAB (COLAB)")
    print("=" * 40)

    # Recording 1
    file1 = record_audio_colab("Recording 1: Speak Normally", 5)
    display(Audio(file1))

    audio1, rate, width = read_wav(file1)
    stats1 = analyze_audio(audio1, rate)
    text1 = transcribe(audio1, rate, width)
    display_stats(stats1, text1, "Recording 1")

    # Recording 2
    file2 = record_audio_colab("Recording 2: Speak Louder/Faster", 5)
    display(Audio(file2))

    audio2, rate, width = read_wav(file2)
    stats2 = analyze_audio(audio2, rate)
    text2 = transcribe(audio2, rate, width)
    display_stats(stats2, text2, "Recording 2")

    compare(stats1, stats2)
    plot_both(stats1, stats2, rate)


main()